# Machine Learning II — Trees, Forests, and Neural Nets
## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Apply random forests and gradient boosted trees** for return prediction
2. **Compare tree-based methods to penalized regression** on the same data
3. **Understand the "no free lunch"** — when each method is best
4. **Read feature importance** outputs and avoid misinterpreting them
5. **Audit AI-generated ML code** — tree depth limits, ensemble size, evaluation metrics

## 📋 TOC
1. [Setup](#setup)  2. [Why Trees](#why)
3. [Pitfall Checklist](#pitfalls)  4. [Random Forest](#rf)
5. [Gradient Boosted Trees](#gbt)  6. [Neural Net Footnote](#nn)
7. [🎯 Challenge: Tree vs Linear](#challenge)
8. [Submission](#submit)  9. [Key Takeaways](#takeaways)

---
## 🛠️ Setup <a id="setup"></a>

In [ ]:
#@title Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize']=[10,5]; plt.rcParams['font.size']=11
import warnings; warnings.filterwarnings('ignore')
print("✅ Loaded")

---
## Why Trees <a id="why"></a>

Linear methods (OLS, Lasso) assume the relationship between $X$ and $r$ is
linear. In finance, this is often wrong:
- Momentum's effect is non-linear (huge wins for top-decile, modest for mid)
- Size effect saturates (mega-caps and small-caps behave differently)
- Feature interactions matter (value × momentum > value + momentum)

**Trees capture non-linearity and interactions automatically.** Their downside:
prone to overfitting unless you constrain depth or use ensembles.

---
## 🛡️ Pitfall Checklist <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---------|-----------------|-------------------|
| 1 | **Unconstrained tree depth** | Trees memorize the training set | Cap depth ≤ 10; require min samples per leaf |
| 2 | **Too few trees in the ensemble** | Random forest needs 100+ trees for stable predictions | Plot performance vs n_estimators |
| 3 | **Feature importance ≠ predictive value** | Importance can be high for noise features in small samples | Compare to a permutation-importance test |
| 4 | **Test set contamination via feature engineering** | Same as Lecture 21 but worse with trees because they're flexible | Build features, then split |
| 5 | **Confusing classification with regression** | Some sklearn calls default to classification when you wanted regression | Always set `RandomForestRegressor` explicitly |

---
## Random Forest <a id="rf"></a>

A **random forest** is an ensemble of decision trees. Each tree:
1. Is trained on a bootstrap sample of the training set
2. At each split, considers a random subset of features

The prediction is the average across trees. **Key hyperparameters:**
- `n_estimators` — number of trees (100-500 typical)
- `max_depth` — depth of each tree (5-15 typical)
- `min_samples_leaf` — minimum observations per leaf (10-100)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
np.random.seed(42)

# Simulate non-linear: y depends on x1 * x2 + sin(x3)
N, K = 1000, 10
X = np.random.normal(0, 1, (N, K))
y = X[:, 0] * X[:, 1] + np.sin(X[:, 2]) + np.random.normal(0, 0.5, N)

X_tr, X_te = X[:800], X[800:]
y_tr, y_te = y[:800], y[800:]

ols = LinearRegression().fit(X_tr, y_tr)
rf  = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=20, random_state=42).fit(X_tr, y_tr)

print(f"OLS OOS R²:  {ols.score(X_te, y_te):.3f}")
print(f"RF  OOS R²:  {rf.score(X_te, y_te):.3f}")
print(f"\nTrees can detect non-linear/interaction effects OLS cannot.")

---
## Gradient Boosted Trees <a id="gbt"></a>

**Gradient boosting** builds trees sequentially, where each tree fits the
residual of the previous trees. Better OOS performance than random forest
when tuned correctly, but more prone to overfit.

Modern implementations: **XGBoost, LightGBM, CatBoost** — all fast and well-tested.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
gbt = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05).fit(X_tr, y_tr)
print(f"GBT OOS R²: {gbt.score(X_te, y_te):.3f}")

---
## Neural Net Footnote <a id="nn"></a>

Deep neural nets exist and can outperform trees on huge datasets. For return
prediction, **the data isn't big enough** for neural nets to consistently
beat tree-based methods.

Gu, Kelly & Xiu (2020) is the standard reference — they compared 11 methods
on US stock returns and found GBT and neural nets approximately tied.

> **💡 Practical note**
>
> For 95% of return-prediction problems, **gradient boosted trees** are the
> right default. Use neural nets only if you have lots of data and a clear
> reason to believe the additional flexibility matters.

---
## 🎯 Challenge: Tree vs Linear <a id="challenge"></a>

> **Setup.** Using simulated data with interactions and non-linearity.

### Q1 — OLS baseline

> **📌 Required:**
> ```python
> ols_oos_r2 = ____    # ols.score(X_te, y_te)
> ```

In [ ]:
ols_oos_r2 = ____
print(f"OLS R²: {ols_oos_r2:.3f}")

### Q2 — Random Forest

> **📌 Required:**
> ```python
> rf_oos_r2 = ____    # rf.score(X_te, y_te) from above
> ```

In [ ]:
rf_oos_r2 = ____
print(f"RF R²: {rf_oos_r2:.3f}")

### Q3 — GBT

> **📌 Required:**
> ```python
> gbt_oos_r2 = ____   # gbt.score(X_te, y_te) from above
> ```

In [ ]:
gbt_oos_r2 = ____
print(f"GBT R²: {gbt_oos_r2:.3f}")

### Q4 — Best method
> **📌 Required:**
> ```python
> best_method_r2 = ____   # max of the three R² above
> ```

In [ ]:
best_method_r2 = ____
print(f"Best method R²: {best_method_r2:.3f}")

### Q5 — Memo

Max 5 sentences. (i) Which method won? (ii) Why does it make sense given the data structure? (iii) What's the risk of using GBT in production?

In [ ]:
MEMO = """Write your memo here."""
print(MEMO)

---
## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL ===
import json, base64, hashlib, datetime as dt
required = ["ols_oos_r2", "rf_oos_r2", "gbt_oos_r2", "best_method_r2", "MEMO"]
missing = [v for v in required if v not in dir()]
if missing: raise NameError(f"\n❌ Missing: {missing}")
payload = {"assignment": "ML_II_AI",
    "ts": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip()}
blob = json.dumps(payload, sort_keys=True)
token = f"UG54::{hashlib.sha256(blob.encode()).hexdigest()[:8]}::{base64.b64encode(blob.encode()).decode()}"
print("="*72); print(token); print("="*72)

---
## 🧠 Key Takeaways <a id="takeaways"></a>
1. **Trees capture non-linearity and interactions automatically.** OLS / Lasso can't.
2. **Random forests are stable;** gradient boosted trees can be better-tuned but riskier.
3. **Neural nets rarely help in finance** because the data isn't big enough.
4. **Cap tree depth; use ensembles.** Single deep trees overfit catastrophically.
5. **AI fits the model in 3 lines. You design the experiment, choose the metric, and resist over-tuning.**